In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Portfolio Projects/Payment Date Prediction(Revamped)/dataset.csv')

In [ ]:
# Parse dates
df['posting_date'] = pd.to_datetime(df['posting_date'], format='mixed')
df['due_in_date'] = pd.to_datetime(df['due_in_date'].astype(str), format='%Y%m%d')
df['clear_date'] = pd.to_datetime(df['clear_date'], format='mixed', errors='coerce')

In [ ]:
# Focus on closed invoices for training
closed = df[df['isOpen'] == 0].copy()
closed['days_late'] = (closed['clear_date'] - closed['due_in_date']).dt.days

In [ ]:
print(f"Training data: {len(closed)} closed invoices")
print(f"Target variable (days_late) distribution:")
print(closed['days_late'].describe())

Training data: 40000 closed invoices
Target variable (days_late) distribution:
count    40000.000000
mean         0.837700
std         10.831701
min        -89.000000
25%         -3.000000
50%          0.000000
75%          2.000000
max        204.000000
Name: days_late, dtype: float64


In [ ]:
def engineer_features(df, history_df=None):
    """
    Engineer features for payment prediction.

    If history_df is provided, use it to calculate historical features.
    Otherwise, use self-referential features (for training).
    """
    features = df.copy()

    # Invoice characteristics
    features['invoice_amount'] = features['total_open_amount']
    features['amount_log'] = np.log1p(features['total_open_amount'])

    # Temporal features
    features['month'] = features['posting_date'].dt.month
    features['day_of_week'] = features['posting_date'].dt.dayofweek
    features['is_month_end'] = features['posting_date'].dt.day > 25
    features['is_year_end'] = features['posting_date'].dt.month == 12

    # Payment terms encoding (extract numeric part if possible)
    features['payment_terms_encoded'] = pd.factorize(features['cust_payment_terms'])[0]

    # Business unit
    features['business_code_encoded'] = pd.factorize(features['business_code'])[0]

    # Customer historical behavior (critical feature)
    if history_df is not None:
        # Use provided history
        cust_history = history_df.groupby('cust_number').agg({
            'days_late': ['mean', 'std', 'count', 'min', 'max'],
            'total_open_amount': ['mean', 'sum']
        })
        cust_history.columns = ['cust_avg_days_late', 'cust_std_days_late',
                                'cust_invoice_count', 'cust_min_late', 'cust_max_late',
                                'cust_avg_amount', 'cust_total_amount']
        cust_history = cust_history.reset_index()
        features = features.merge(cust_history, on='cust_number', how='left')
    else:
        # Self-referential (for training - will use leave-one-out in practice)
        cust_history = features.groupby('cust_number').agg({
            'days_late': ['mean', 'std', 'count', 'min', 'max'],
            'total_open_amount': ['mean', 'sum']
        })
        cust_history.columns = ['cust_avg_days_late', 'cust_std_days_late',
                                'cust_invoice_count', 'cust_min_late', 'cust_max_late',
                                'cust_avg_amount', 'cust_total_amount']
        cust_history = cust_history.reset_index()
        features = features.merge(cust_history, on='cust_number', how='left')

    # Fill missing customer history (new customers)
    features['cust_avg_days_late'] = features['cust_avg_days_late'].fillna(features['days_late'].mean() if 'days_late' in features.columns else 0)
    features['cust_std_days_late'] = features['cust_std_days_late'].fillna(features['days_late'].std() if 'days_late' in features.columns else 10)
    features['cust_invoice_count'] = features['cust_invoice_count'].fillna(1)
    features['cust_min_late'] = features['cust_min_late'].fillna(0)
    features['cust_max_late'] = features['cust_max_late'].fillna(0)
    features['cust_avg_amount'] = features['cust_avg_amount'].fillna(features['total_open_amount'])
    features['cust_total_amount'] = features['cust_total_amount'].fillna(features['total_open_amount'])

    return features

In [ ]:
closed_featured = engineer_features(closed)

In [ ]:
closed_featured.head()

,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,...,is_year_end,payment_terms_encoded,business_code_encoded,cust_avg_days_late,cust_std_days_late,cust_invoice_count,cust_min_late,cust_max_late,cust_avg_amount,cust_total_amount
0,U001,200769623,WAL-MAR corp,2020-02-11,2020,1930438491,2020-01-26,20200125,20200126,2020-02-10,...,False,0,0,-2.366246,6.031129,9581,-15,92,22583.260607,2.163702e+08
1,U001,200980828,BEN E,2019-08-08,2019,1929646410,2019-07-22,20190722,20190722,2019-08-11,...,False,1,0,-3.203704,17.960522,54,-30,120,32434.321481,1.751453e+06
2,U001,200792734,MDV/ trust,2019-12-30,2019,1929873765,2019-09-14,20190914,20190914,2019-09-29,...,False,2,0,1.962025,11.607095,158,-2,92,36966.015190,5.840630e+06
3,U001,200769623,WAL-MAR foundation,2019-11-25,2019,1930147974,2019-11-13,20191113,20191113,2019-11-28,...,False,0,0,-2.366246,6.031129,9581,-15,92,22583.260607,2.163702e+08
4,CA02,140106181,THE corporation,2019-12-04,2019,2960581231,2019-09-20,20190920,20190920,2019-10-04,...,False,3,1,62.980000,22.376476,100,-5,171,29145.989700,2.914599e+06


In [ ]:
# Define feature columns
feature_cols = [
    'amount_log', 'month', 'day_of_week', 'is_month_end', 'is_year_end',
    'payment_terms_encoded', 'business_code_encoded',
    'cust_avg_days_late', 'cust_std_days_late', 'cust_invoice_count',
    'cust_min_late', 'cust_max_late', 'cust_avg_amount'
]

print(f"Features: {feature_cols}")

Features: ['amount_log', 'month', 'day_of_week', 'is_month_end', 'is_year_end', 'payment_terms_encoded', 'business_code_encoded', 'cust_avg_days_late', 'cust_std_days_late', 'cust_invoice_count', 'cust_min_late', 'cust_max_late', 'cust_avg_amount']


In [ ]:
# Prepare data
X = closed_featured[feature_cols].values
y = closed_featured['days_late'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train model for point prediction
model_mean = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model_mean.fit(X_train, y_train)

GradientBoostingRegressor(max_depth=5, random_state=42)

In [ ]:
# Train models for prediction intervals (10th and 90th percentiles)
model_lower = GradientBoostingRegressor(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    loss='quantile', alpha=0.1, random_state=42
)
model_upper = GradientBoostingRegressor(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    loss='quantile', alpha=0.9, random_state=42
)
model_lower.fit(X_train, y_train)
model_upper.fit(X_train, y_train)

# Predictions
y_pred = model_mean.predict(X_test)
y_lower = model_lower.predict(X_test)
y_upper = model_upper.predict(X_test)

In [ ]:
y_upper

array([ 5.46415655, -4.9091226 ,  1.68558117, ...,  2.87726254,
       -4.82518524,  8.63174949])

In [ ]:
y_pred

array([ 2.74983961, -6.69637507, -2.65489359, ...,  1.48179947,
       -7.24919942,  5.64709726])

In [ ]:
y_lower

array([ -0.87345137, -10.13300458,  -4.00002656, ...,  -0.18119839,
        -9.98675189,  -1.54090933])

In [ ]:
# Evaluate
rmse = np.sqrt(np.mean((y_test - y_pred)**2))
mae = np.mean(np.abs(y_test - y_pred))
within_3_days = np.mean(np.abs(y_test - y_pred) <= 3)
coverage = np.mean((y_test >= y_lower) & (y_test <= y_upper))

print("=== Model Performance ===")
print(f"RMSE: {rmse:.2f} days")
print(f"MAE: {mae:.2f} days")
print(f"Within ±3 days: {within_3_days*100:.1f}%")
print(f"80% Prediction Interval Coverage: {coverage*100:.1f}%")

=== Model Performance ===
RMSE: 6.57 days
MAE: 2.69 days
Within ±3 days: 78.7%
80% Prediction Interval Coverage: 76.2%


In [ ]:
# ## 4. THE KEY INNOVATION: Decision Framework
#
# This is where we transform predictions into decisions.
#
# ### The Decision Problem
#
# For each invoice, we calculate:
#
# ```
# Expected Value of Intervention =
#     P(late) × P(responds to call) × Days_Accelerated × Daily_Capital_Cost × Invoice_Amount
#     - Cost_of_Call
# ```
#
# Then we rank invoices by expected value and allocate collection capacity accordingly.

# %%
class DecisionEngine:
    """
    Transform predictions into prioritized action recommendations.
    """

    def __init__(
        self,
        daily_capital_cost_rate=0.0003,  # ~10% annual cost of capital
        cost_per_call=15.0,              # Staff time + overhead
        avg_days_accelerated=3.0,         # Estimated effect of intervention
        daily_capacity=20                 # Calls per day
    ):
        self.daily_capital_cost = daily_capital_cost_rate
        self.call_cost = cost_per_call
        self.days_accelerated = avg_days_accelerated
        self.capacity = daily_capacity

    def calculate_expected_value(
        self,
        invoice_amount,
        predicted_days_late,
        prediction_lower,
        prediction_upper,
        customer_responsiveness=0.5
    ):
        """
        Calculate expected value of making a collection call.

        Returns:
            expected_value: Expected $ benefit of calling
            components: Breakdown of calculation
        """
        # Probability of being late (using prediction interval)
        # If upper bound is negative, very unlikely to be late
        if prediction_upper < 0:
            p_late = 0.05  # Small baseline probability
        elif prediction_lower > 5:
            p_late = 0.95  # Almost certainly late
        else:
            # Linear interpolation based on where 0 falls in interval
            interval_width = prediction_upper - prediction_lower
            if interval_width > 0:
                p_late = max(0, min(1, (prediction_upper - 0) / interval_width))
            else:
                p_late = 0.5

        # Expected benefit if we call and customer responds
        benefit_if_success = (
            self.days_accelerated *
            self.daily_capital_cost *
            invoice_amount
        )

        # Expected value calculation
        expected_benefit = p_late * customer_responsiveness * benefit_if_success
        expected_value = expected_benefit - self.call_cost

        return expected_value, {
            'p_late': p_late,
            'customer_responsiveness': customer_responsiveness,
            'benefit_if_success': benefit_if_success,
            'expected_benefit': expected_benefit,
            'call_cost': self.call_cost,
            'expected_value': expected_value
        }

    def prioritize_invoices(self, invoices_df):
        """
        Rank invoices by expected value of intervention.

        Returns DataFrame with recommendations.
        """
        results = []

        for idx, row in invoices_df.iterrows():
            ev, components = self.calculate_expected_value(
                invoice_amount=row['invoice_amount'],
                predicted_days_late=row['predicted_days_late'],
                prediction_lower=row['prediction_lower'],
                prediction_upper=row['prediction_upper'],
                customer_responsiveness=row.get('customer_responsiveness', 0.5)
            )

            results.append({
                'invoice_id': row.get('invoice_id', idx),
                'customer': row.get('cust_number', 'Unknown'),
                'invoice_amount': row['invoice_amount'],
                'predicted_days_late': row['predicted_days_late'],
                'prediction_interval': f"[{row['prediction_lower']:.0f}, {row['prediction_upper']:.0f}]",
                'p_late': components['p_late'],
                'expected_value': ev,
                'recommendation': 'CALL' if ev > 0 else 'SKIP'
            })

        results_df = pd.DataFrame(results)
        results_df = results_df.sort_values('expected_value', ascending=False)

        # Mark top N as high priority
        results_df['priority'] = 'LOW'
        results_df.iloc[:self.capacity, results_df.columns.get_loc('priority')] = 'HIGH'

        return results_df

# Initialize decision engine
engine = DecisionEngine(
    daily_capital_cost_rate=0.0003,  # ~10% annual
    cost_per_call=15.0,
    avg_days_accelerated=3.0,
    daily_capacity=20
)

print("Decision Engine initialized with:")
print(f"  - Daily capital cost: {engine.daily_capital_cost*100:.2f}%")
print(f"  - Cost per call: ${engine.call_cost}")
print(f"  - Assumed days accelerated: {engine.days_accelerated}")
print(f"  - Daily capacity: {engine.capacity} calls")

Decision Engine initialized with:
  - Daily capital cost: 0.03%
  - Cost per call: $15.0
  - Assumed days accelerated: 3.0
  - Daily capacity: 20 calls


In [ ]:
# ## 5. Apply Decision Framework to Test Set
#
# Let's see how the decision engine prioritizes invoices.

# %%
# Create test set with predictions
test_df = closed_featured.iloc[X_train.shape[0]:].copy()
test_df['predicted_days_late'] = y_pred
test_df['prediction_lower'] = y_lower
test_df['prediction_upper'] = y_upper
test_df['invoice_amount'] = test_df['total_open_amount']

# Calculate customer responsiveness based on historical variance
# (customers with consistent behavior are more predictable)
test_df['customer_responsiveness'] = np.clip(
    1 - (test_df['cust_std_days_late'] / 20),
    0.2, 0.8
)

# Apply decision engine
recommendations = engine.prioritize_invoices(test_df)

print("=== TOP 10 PRIORITY INVOICES ===")
print(recommendations.head(10).to_string(index=False))

print("\n=== RECOMMENDATION SUMMARY ===")
print(recommendations['recommendation'].value_counts())
print(f"\nTotal Expected Value from calls: ${recommendations[recommendations['expected_value'] > 0]['expected_value'].sum():,.2f}")

=== TOP 10 PRIORITY INVOICES ===
  invoice_id  customer  invoice_amount  predicted_days_late prediction_interval   p_late  expected_value recommendation priority
1930303665.0 200002965       439919.82             8.756068            [-5, 19] 0.797890      236.420357           CALL     HIGH
2960530661.0 140104440       302720.89             1.778944             [-0, 2] 0.999895      192.345839           CALL     HIGH
2960601840.0 140104409       299724.84             1.042753             [-0, 3] 0.999921      172.605897           CALL     HIGH
2960567060.0 140104440       269908.59             1.464908             [-0, 3] 0.999927      169.877226           CALL     HIGH
2960528400.0 140104440       292496.26             4.078194             [-1, 8] 0.903898      166.108264           CALL     HIGH
2960539983.0 140104423       285856.06             3.246771              [0, 7] 1.000000      161.816684           CALL     HIGH
2960533904.0 140104409       285203.34             1.985731     

In [ ]:
# ## 6. Business Impact Analysis
#
# What's the ROI of this system compared to naive approaches?

# %%
# Scenario 1: Random calling (baseline)
random_sample = recommendations.sample(n=engine.capacity, random_state=42)
random_ev = random_sample['expected_value'].sum()

# Scenario 2: Call highest invoice amounts (naive prioritization)
by_amount = recommendations.nlargest(engine.capacity, 'invoice_amount')
amount_ev = by_amount['expected_value'].sum()

# Scenario 3: Decision engine (our approach)
optimized = recommendations.head(engine.capacity)
optimized_ev = optimized['expected_value'].sum()

print("=== DAILY IMPACT COMPARISON ===")
print(f"Random calling:       ${random_ev:>10,.2f} expected value")
print(f"By invoice amount:    ${amount_ev:>10,.2f} expected value")
print(f"Decision engine:      ${optimized_ev:>10,.2f} expected value")
print(f"\nImprovement over random:  {(abs(optimized_ev/random_ev) - 1)*100:+.1f}%")
print(f"Improvement over amount:  {(abs(optimized_ev/amount_ev) - 1)*100:+.1f}%")

# Annualized impact (assuming 250 working days)
annual_improvement = (optimized_ev - random_ev) * 250
print(f"\n=== ANNUALIZED IMPACT ===")
print(f"Annual improvement over random: ${annual_improvement:,.2f}")

=== DAILY IMPACT COMPARISON ===
Random calling:       $   -123.56 expected value
By invoice amount:    $  2,083.15 expected value
Decision engine:      $  3,063.51 expected value

Improvement over random:  +2379.4%
Improvement over amount:  +47.1%

=== ANNUALIZED IMPACT ===
Annual improvement over random: $796,766.44


In [ ]:
# ## 7. Asymmetric Loss Analysis
#
# Showing why RMSE isn't the right metric.

# %%
# Create asymmetric scenarios
scenarios = []

for idx, row in recommendations.head(100).iterrows():
    actual = y_test[idx] if idx < len(y_test) else 0
    pred = row['predicted_days_late']
    amount = row['invoice_amount']
    error = np.round(actual - pred, decimals=0)

    # Symmetric loss (RMSE)
    symmetric_loss = error ** 2

    # Asymmetric loss (under-prediction costs more for large invoices)
    if error > 0:  # Predicted early, actually late (bad!)
        asymmetric_loss = (error ** 2) * (amount / 10000) * 2
    else:  # Predicted late, actually early (less bad)
        asymmetric_loss = (error ** 2) * (amount / 10000) * 0.5

    scenarios.append({
        'predicted': pred,
        'actual': actual,
        'error': error,
        'invoice_amount': amount,
        'symmetric_loss': symmetric_loss,
        'asymmetric_loss': asymmetric_loss
    })

scenarios_df = pd.DataFrame(scenarios)

print("=== LOSS FUNCTION COMPARISON ===")
print(f"Mean Symmetric Loss:   {scenarios_df['symmetric_loss'].mean():.2f}")
print(f"Mean Asymmetric Loss:  {scenarios_df['asymmetric_loss'].mean():.2f}")

print("\n=== EXAMPLES WHERE THEY DIFFER ===")
print("High invoice, under-predicted (we thought on-time, was late):")
bad_cases = scenarios_df[(scenarios_df['error'] == 3) & (scenarios_df['invoice_amount'] > 50000)]
good_cases = scenarios_df[(scenarios_df['error'] == -3) & (scenarios_df['invoice_amount'] > 50000)]
if len(bad_cases) > 0:
    print(bad_cases[['invoice_amount', 'predicted', 'actual', 'symmetric_loss', 'asymmetric_loss']].head(3).to_string(index=False))
else:
    print("  (No cases in sample)")

print("High invoice, over-predicted (we thought late, was on-time):")

if len(good_cases) > 0:
    print(good_cases[['invoice_amount', 'predicted', 'actual', 'symmetric_loss', 'asymmetric_loss']].head(3).to_string(index=False))
else:
    print("  (No cases in sample)")

=== LOSS FUNCTION COMPARISON ===
Mean Symmetric Loss:   14.42
Mean Asymmetric Loss:  440.06

=== EXAMPLES WHERE THEY DIFFER ===
High invoice, under-predicted (we thought on-time, was late):
 invoice_amount  predicted  actual  symmetric_loss  asymmetric_loss
      172627.88   0.432692       3             9.0       310.730184
      170503.18   1.557396       5             9.0       306.905724
      115524.36   0.721057       4             9.0       207.943848
High invoice, over-predicted (we thought late, was on-time):
 invoice_amount  predicted  actual  symmetric_loss  asymmetric_loss
      226222.58   2.074250      -1             9.0       101.800161
      194959.36   3.273088       0             9.0        87.731712
      176020.51   3.058860       0             9.0        79.209229


In [ ]:
# ## 8. Output: Daily Action Queue
#
# What the collections team actually sees.

# %%
def generate_daily_queue(recommendations, date="2024-01-15"):
    """Generate the daily work queue for the collection team."""

    queue = recommendations[recommendations['priority'] == 'HIGH'].copy()

    print(f"=" * 60)
    print(f"COLLECTION QUEUE FOR {date}")
    print(f"=" * 60)
    print(f"Total calls scheduled: {len(queue)}")
    print(f"Total invoice value at risk: ${queue['invoice_amount'].sum():,.2f}")
    print(f"Expected value of today's calls: ${queue['expected_value'].sum():,.2f}")
    print(f"=" * 60)
    print()

    for i, (_, row) in enumerate(queue.iterrows(), 1):
        print(f"{i:2d}. Customer: {row['customer']}")
        print(f"    Invoice: ${row['invoice_amount']:,.2f}")
        print(f"    Predicted: {row['predicted_days_late']:.0f} days late (80% CI: {row['prediction_interval']})")
        print(f"    Expected Value: ${row['expected_value']:.2f}")
        print()

    return queue

# Generate sample queue
daily_queue = generate_daily_queue(recommendations)

COLLECTION QUEUE FOR 2024-01-15
Total calls scheduled: 20
Total invoice value at risk: $5,472,896.94
Expected value of today's calls: $3,063.51

 1. Customer: 200002965
    Invoice: $439,919.82
    Predicted: 9 days late (80% CI: [-5, 19])
    Expected Value: $236.42

 2. Customer: 140104440
    Invoice: $302,720.89
    Predicted: 2 days late (80% CI: [-0, 2])
    Expected Value: $192.35

 3. Customer: 140104409
    Invoice: $299,724.84
    Predicted: 1 days late (80% CI: [-0, 3])
    Expected Value: $172.61

 4. Customer: 140104440
    Invoice: $269,908.59
    Predicted: 1 days late (80% CI: [-0, 3])
    Expected Value: $169.88

 5. Customer: 140104440
    Invoice: $292,496.26
    Predicted: 4 days late (80% CI: [-1, 8])
    Expected Value: $166.11

 6. Customer: 140104423
    Invoice: $285,856.06
    Predicted: 3 days late (80% CI: [0, 7])
    Expected Value: $161.82

 7. Customer: 140104409
    Invoice: $285,203.34
    Predicted: 2 days late (80% CI: [-0, 4])
    Expected Value: $15